# WaveGrip: Loading & Preprocessing Pipeline
## GRABMyo: Build Wrist sEMG Datasets (Full Dataset)

Tristan Au

This notebook runs the steps from `grabmyo_sample_datapoint.ipynb` on **all 15,351 trials** of GRABMyo (3 sessions × 43 participants × 17 gestures × 7 trials). It uses only the **12 wrist channels** (W1–W12) and builds three datasets, then checks each one against a direct recomputation.

| Dataset | Pipeline | Window | Output |
|---|---|---|---|
| **Raw** | load raw sEMG (int16 ADC counts + gain/baseline) | the whole trial | `ProcessedDatasets/raw_wrist/grabmyo_wrist_raw.h5` |
| **DWT** | load raw → DWT (`db4`, 5 levels, `periodization`) → keep A5, D5–D2 (0–512 Hz) | **the whole trial: 10240 samples = 5 s**, one DWT per channel per trial, no sliding window | `ProcessedDatasets/dwt_wrist/grabmyo_wrist_dwt.h5` |
| **PSD** | load raw → Welch PSD (FFT) → keep the 10–500 Hz bins | the whole trial, averaged over 1024-sample (0.5 s) Hann segments with 50 % overlap | `ProcessedDatasets/psd_wrist/grabmyo_wrist_psd.h5` |

**Side Notes:**

**No filtering is added.** The released GRABMyo signals already carry the authors' 4th-order Butterworth 10–500 Hz band-pass and 60 Hz notch (Pradhan et al., *Scientific Data* 9:733 (2022); checked on the spectrum in `grabmyo_sample_datapoint.ipynb`, Section 2). All three datasets start from the signal as released.

**Window size.** Each DWT covers one full 5 s trial, the same window used in `grabmyo_sample_datapoint.ipynb`.

**Band selection.** Sampling at 2048 Hz means every representation spans 0–1024 Hz (the Nyquist frequency), but the released data only carries sEMG in 10–500 Hz: above 500 Hz is the roll-off of the authors' 4th-order Butterworth plus noise. So the DWT and PSD datasets store only the part of each representation that covers the passband (details in Sections 5 and 6). This is done by **choosing which coefficients (DWT) or bins (PSD) to keep**, not by filtering the signal again: a brick-wall ("ideal") band-pass cannot be built in the time domain, and approximating one causes ringing that smears sEMG bursts in time. The Raw dataset keeps the full recording, so any band can be re-derived later.

**Required Storage.** Paths are relative to the project folder (the parent of `Preprocessing/`, where this notebook runs from): source data is read from `Dataset/physionet.org/files/grabmyo/1.1.0/` and outputs are written to `ProcessedDatasets/`. Both folders are git-ignored. The three outputs need about 8 GB of free disk (Raw ≈ 3.8 GB, DWT ≈ 3.8 GB, PSD ≈ 0.2 GB).

## 0. Configuration

In [1]:
import os
import re
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pywt
import wfdb
from scipy import signal

# Source dataset and outputs live in the project folder (both git-ignored); run from Preprocessing/
PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "Dataset" / "physionet.org" / "files" / "grabmyo" / "1.1.0"
OUT_ROOT = PROJECT_ROOT / "ProcessedDatasets"
OUT_PATHS = {
    "Raw": OUT_ROOT / "raw_wrist" / "grabmyo_wrist_raw.h5",
    "DWT": OUT_ROOT / "dwt_wrist" / "grabmyo_wrist_dwt.h5",
    "PSD": OUT_ROOT / "psd_wrist" / "grabmyo_wrist_psd.h5",
}
assert (DATA_ROOT / "RECORDS").exists(), f"GRABMyo dataset not found at {DATA_ROOT}"

FS = 2048
N_SAMPLES = 10240                       # 5 s per trial
WRIST = [f"W{i}" for i in range(1, 13)]
N_CH = len(WRIST)

# No filtering: the released data already has a 10-500 Hz band-pass + 60 Hz notch (Pradhan et al., 2022)

# DWT: identical to grabmyo_sample_datapoint.ipynb
DWT_WINDOW = N_SAMPLES                  # window = the whole 5 s trial (10240 samples); no sliding window
WAVELET, DWT_LEVEL, DWT_MODE = "db4", 5, "periodization"
DWT_BANDS = [f"A{DWT_LEVEL}"] + [f"D{j}" for j in range(DWT_LEVEL, 0, -1)]      # A5, D5, ..., D1
DWT_LENGTHS = [DWT_WINDOW // 2**DWT_LEVEL] + [DWT_WINDOW // 2**j for j in range(DWT_LEVEL, 0, -1)]
DWT_DROP = ["D1"]                       # 512-1024 Hz: above the released 10-500 Hz passband, not stored
DWT_KEEP = [b for b in DWT_BANDS if b not in DWT_DROP]                        # A5, D5, D4, D3, D2 (0-512 Hz)

# PSD: Welch = average of FFT periodograms of 50%-overlapping Hann-windowed segments of the whole trial
PSD_NPERSEG = 1024                      # 0.5 s segments -> 2 Hz resolution, 513 bins (0-1024 Hz)
PSD_NOVERLAP = PSD_NPERSEG // 2
PSD_BAND = (10.0, 500.0)                # Hz: store only the bins in the released passband
PSD_FREQS_ALL = np.fft.rfftfreq(PSD_NPERSEG, 1 / FS)                               # 513 bins, 0-1024 Hz
PSD_KEEP = (PSD_FREQS_ALL >= PSD_BAND[0]) & (PSD_FREQS_ALL <= PSD_BAND[1])
PSD_FREQS = PSD_FREQS_ALL[PSD_KEEP]                                                # 246 bins, 10-500 Hz
N_FREQ = len(PSD_FREQS)

# Batching / parallelism
BATCH_SIZE = 64                         # trials held in RAM at once
N_THREADS = os.cpu_count()
WORK_CHUNK = 2                          # trials per thread task
OVERWRITE = True                        # rebuild output files that already exist

print(f"DWT window: {DWT_WINDOW} samples = {DWT_WINDOW / FS:g} s (whole trial); "
      f"{WAVELET}, {DWT_LEVEL} levels, {DWT_MODE}; stored bands: {', '.join(DWT_KEEP)}")
print(f"PSD: Welch, {PSD_NPERSEG}-sample Hann segments, {PSD_NOVERLAP}-sample overlap; "
      f"stored: {N_FREQ} bins, {PSD_FREQS[0]:g}-{PSD_FREQS[-1]:g} Hz")
print(f"{N_THREADS} threads, batch size {BATCH_SIZE}, "
      f"free disk: {shutil.disk_usage(OUT_ROOT.parent).free / 1e9:.0f} GB")

DWT window: 10240 samples = 5 s (whole trial); db4, 5 levels, periodization; stored bands: A5, D5, D4, D3, D2
PSD: Welch, 1024-sample Hann segments, 512-sample overlap; stored: 246 bins, 10-500 Hz
8 threads, batch size 64, free disk: 176 GB


## 1. Record index

`RECORDS` lists most records twice, so duplicates are removed. The index is sorted by session → participant → gesture → trial, and that row order is the same in every output dataset.

In [2]:
_REC_RE = re.compile(r"session(\d+)_participant(\d+)_gesture(\d+)_trial(\d+)$")

rels = sorted({ln.strip() for ln in (DATA_ROOT / "RECORDS").read_text().splitlines() if ln.strip()})
index = pd.DataFrame([(r, *map(int, _REC_RE.search(r).groups())) for r in rels],
                     columns=["path", "session", "participant", "gesture", "trial"])
index = index.sort_values(["session", "participant", "gesture", "trial"], ignore_index=True)
N_REC = len(index)

missing = [p for p in index.path if not (DATA_ROOT / f"{p}.dat").exists()]
assert not missing, f"{len(missing)} missing .dat files, e.g. {missing[:3]}"
print(f"{N_REC} records: {index.session.nunique()} sessions, {index.participant.nunique()} participants, "
      f"{index.gesture.nunique()} gestures, {index.trial.nunique()} trials")
index.head()

15351 records: 3 sessions, 43 participants, 17 gestures, 7 trials


,path,session,participant,gesture,trial
0,Session1/session1_participant1/session1_partic...,1,1,1,1
1,Session1/session1_participant1/session1_partic...,1,1,1,2
2,Session1/session1_participant1/session1_partic...,1,1,1,3
3,Session1/session1_participant1/session1_partic...,1,1,1,4
4,Session1/session1_participant1/session1_partic...,1,1,1,5


## 2. Raw reader

GRABMyo uses WFDB **format 16**: little-endian int16 samples, interleaved across all 32 channels. The `.hea` header gives each channel's `gain(baseline)/mV`, and physical value = (count − baseline) / gain [[WFDB signal & header specs](https://www.physionet.org/physiotools/wag/signal-5.htm)].

This reader parses the header with one regex and loads the `.dat` file with `np.fromfile`. It keeps only the 12 wrist channels as int16 counts; conversion to mV happens inside the worker threads.

In [3]:
_HEA_SIG = re.compile(r"^\S+\s+16\s+([-+\d.eE]+)\((-?\d+)\)/mV\s+\S+\s+\S+\s+\S+\s+\S+\s+\S+\s+(\S+)$")


def read_record(rel):
    # Returns int16 counts (12, N), gain (12,), baseline (12,) for the wrist channels of one trial
    base = DATA_ROOT / rel
    header = Path(f"{base}.hea").read_text().splitlines()
    n_sig, fs, n = (int(v) for v in header[0].split()[1:4])
    if fs != FS or n != N_SAMPLES:
        raise ValueError(f"{rel}: unexpected fs={fs} / n={n}")
    gain, baseline, names = [], [], []
    for ln in header[1:1 + n_sig]:
        m = _HEA_SIG.match(ln.strip())
        if m is None:
            raise ValueError(f"{rel}: unexpected header line {ln!r}")
        gain.append(float(m.group(1))); baseline.append(int(m.group(2))); names.append(m.group(3))
    idx = [names.index(w) for w in WRIST]
    counts = np.fromfile(f"{base}.dat", dtype="<i2", count=n * n_sig).reshape(n, n_sig)
    return np.ascontiguousarray(counts[:, idx].T), np.asarray(gain)[idx], np.asarray(baseline)[idx]


def read_batch(batch_rels, pool):
    # Threaded read of a batch -> counts (B, 12, N) int16, gain (B, 12) f8, baseline (B, 12) i4
    out = list(pool.map(read_record, batch_rels))
    return (np.stack([o[0] for o in out]),
            np.stack([o[1] for o in out]),
            np.stack([o[2] for o in out]).astype(np.int32))


def to_mv(counts, gain, baseline):
    return (counts - baseline[..., None]) / gain[..., None]        # float64, mV

### 2.1 Reader check

Confirms that the reader returns exactly what `wfdb.rdrecord` does, on 16 random trials.

In [4]:
def dwt_worker(counts, gain, baseline):
    # mV -> DWT over the whole trial (DWT_WINDOW samples); outputs float32 for the kept bands [A5, D5, ..., D2]
    coeffs = pywt.wavedec(to_mv(counts, gain, baseline), WAVELET, mode=DWT_MODE, level=DWT_LEVEL, axis=-1)
    return [c.astype(np.float32) for band, c in zip(DWT_BANDS, coeffs) if band in DWT_KEEP]


def psd_worker(counts, gain, baseline):
    # mV -> Welch PSD (mV^2/Hz) over the whole trial; outputs float32 (B, 12, 246), the 10-500 Hz bins
    _, p = signal.welch(to_mv(counts, gain, baseline), fs=FS, nperseg=PSD_NPERSEG,
                        noverlap=PSD_NOVERLAP, window="hann", axis=-1)
    return [p[..., PSD_KEEP].astype(np.float32)]


def run_threaded(worker, counts, gain, baseline, pool, chunk=WORK_CHUNK):
    # Split a batch into small trial chunks, run them on the thread pool, re-join outputs
    parts = np.array_split(np.arange(len(counts)), -(-len(counts) // chunk))
    results = list(pool.map(lambda ix: worker(counts[ix], gain[ix], baseline[ix]), parts))
    return [np.concatenate(arrs) for arrs in zip(*results)]

In [5]:
sample_rels = index.path.sample(16, random_state=0).tolist()
with ThreadPoolExecutor(N_THREADS) as pool:
    counts, gain, baseline = read_batch(sample_rels, pool)

for k, rel in enumerate(sample_rels):
    rec = wfdb.rdrecord(str(DATA_ROOT / rel), channel_names=WRIST)
    assert rec.sig_name == WRIST
    np.testing.assert_allclose(to_mv(counts[k], gain[k], baseline[k]), rec.p_signal.T, rtol=0, atol=1e-12)
print(f"Reader output identical to wfdb.rdrecord for all {len(sample_rels)} trials")
del counts, gain, baseline

Reader output identical to wfdb.rdrecord for all 16 trials


## 3. Dataset writer

`build_dataset` streams all trials through read → (process) → write, one batch of `BATCH_SIZE` trials at a time, into pre-allocated HDF5 datasets chunked one trial per chunk. Each file is written as `*.h5.partial` and renamed only when complete, so an interrupted run never leaves a truncated file under the final name. Every file stores the same `session`, `participant`, `gesture`, `trial` and `record` index arrays in the same row order.

In [6]:
def write_common_metadata(f, description):
    f.attrs.update(description=description, source="GRABMyo v1.1.0 (PhysioNet)", fs=FS,
                   n_samples=N_SAMPLES, channels=WRIST, row_order="session, participant, gesture, trial",
                   filter="none added; source data already has a 4th-order Butterworth "
                          "10-500 Hz band-pass and a 60 Hz notch (Pradhan et al., 2022)")
    for col in ["session", "participant", "gesture", "trial"]:
        f.create_dataset(col, data=index[col].to_numpy(np.uint8))
    f.create_dataset("record", data=index.path.to_numpy(dtype=object), dtype=h5py.string_dtype())


def build_dataset(name, init_file, write_batch, worker=None, batch_size=BATCH_SIZE):
    # Stream all trials through read -> (process) -> write, one batch at a time
    out_path = OUT_PATHS[name]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and not OVERWRITE:
        raise FileExistsError(out_path)
    tmp_path = out_path.with_name(out_path.name + ".partial")
    tmp_path.unlink(missing_ok=True)
    rel_list = index.path.tolist()
    n_batches = -(-N_REC // batch_size)
    t0 = time.perf_counter()
    with ThreadPoolExecutor(N_THREADS) as pool, h5py.File(tmp_path, "w") as f:
        init_file(f)
        for b, i0 in enumerate(range(0, N_REC, batch_size)):
            sl = slice(i0, min(i0 + batch_size, N_REC))
            counts, gain, baseline = read_batch(rel_list[sl], pool)
            out = run_threaded(worker, counts, gain, baseline, pool) if worker else None
            write_batch(f, sl, counts, gain, baseline, out)
            if (b + 1) % 40 == 0 or b + 1 == n_batches:
                print(f"  [{name}] batch {b + 1}/{n_batches}  {time.perf_counter() - t0:6.1f} s")
    os.replace(tmp_path, out_path)
    print(f"[{name}] saved {out_path} ({out_path.stat().st_size / 1e9:.2f} GB)")
    return out_path


paths = {}

## 4. Raw dataset

The raw signals are stored **losslessly as the original int16 ADC counts**, together with each channel's gain and baseline. Convert to mV with `mV = (emg − baseline[..., None]) / gain[..., None]`.

| HDF5 key | Shape | dtype |
|---|---|---|
| `emg` | (15351, 12, 10240) | int16 (ADC counts) |
| `gain` | (15351, 12) | float64 (counts per mV) |
| `baseline` | (15351, 12) | int32 (counts) |
| `session`, `participant`, `gesture`, `trial`, `record` | (15351,) | uint8 / str |

In [7]:
def raw_init(f):
    write_common_metadata(f, "Raw GRABMyo wrist sEMG as int16 ADC counts")
    f.attrs["units"] = "ADC counts; mV = (emg - baseline[..., None]) / gain[..., None]"
    f.create_dataset("emg", (N_REC, N_CH, N_SAMPLES), "int16", chunks=(1, N_CH, N_SAMPLES))
    f.create_dataset("gain", (N_REC, N_CH), "float64")
    f.create_dataset("baseline", (N_REC, N_CH), "int32")


def raw_write(f, sl, counts, gain, baseline, out):
    f["emg"][sl] = counts
    f["gain"][sl] = gain
    f["baseline"][sl] = baseline


paths["Raw"] = build_dataset("Raw", raw_init, raw_write)

  [Raw] batch 40/240    23.8 s


  [Raw] batch 80/240    45.8 s


  [Raw] batch 120/240    65.7 s


  [Raw] batch 160/240    84.6 s


  [Raw] batch 200/240   103.7 s


  [Raw] batch 240/240   126.3 s
[Raw] saved C:\Users\trist\Desktop\Research\Wrist-based EMG\Live\ProcessedDatasets\raw_wrist\grabmyo_wrist_raw.h5 (3.78 GB)


## 5. DWT dataset

Each trial goes through mV conversion → `db4` DWT with 5 levels in `periodization` mode, computed over the **whole 5 s trial (10240 samples)** for each channel. Periodization gives exactly N/2^j coefficients per level. The coefficients are stored as float32.

**Note:** Only A5 and D5–D2 (0–512 Hz) are stored; D1 (512–1024 Hz) is dropped to only capture between 0-500 Hz (band selection).

| HDF5 key | Shape | Band | Time per coefficient |
|---|---|---|---|
| `dwt/A5` | (15351, 12, 320) | 0–32 Hz | 15.6 ms |
| `dwt/D5` | (15351, 12, 320) | 32–64 Hz | 15.6 ms |
| `dwt/D4` | (15351, 12, 640) | 64–128 Hz | 7.8 ms |
| `dwt/D3` | (15351, 12, 1280) | 128–256 Hz | 3.9 ms |
| `dwt/D2` | (15351, 12, 2560) | 256–512 Hz | 1.95 ms |

The window, wavelet and band choices are stored as file attributes (`window_samples`, `window_s`, `wavelet`, `level`, `mode`, `bands`, `bands_dropped`). To rebuild a trial's signal restricted to 0–512 Hz (mV), pass zeros for D1: `pywt.waverec([dwt[b][i] for b in bands] + [np.zeros((12, 5120))], "db4", mode="periodization", axis=-1)`.

In [8]:
def dwt_init(f):
    write_common_metadata(f, "GRABMyo wrist sEMG: DWT coefficients of the signal as released")
    f.attrs.update(units="mV", window_samples=DWT_WINDOW, window_s=DWT_WINDOW / FS,
                   windowing="whole trial: one DWT per channel per trial, no sliding window",
                   wavelet=WAVELET, level=DWT_LEVEL, mode=DWT_MODE, bands=DWT_KEEP, bands_dropped=DWT_DROP,
                   band_selection="D1 (512-1024 Hz) not stored: above the released 10-500 Hz passband")
    g = f.create_group("dwt")
    for band, L in zip(DWT_BANDS, DWT_LENGTHS):
        if band not in DWT_KEEP:
            continue
        d = g.create_dataset(band, (N_REC, N_CH, L), "float32", chunks=(1, N_CH, L))
        j = int(band[1:])
        d.attrs["freq_range_Hz"] = (0.0, FS / 2**(j + 1)) if band[0] == "A" else (FS / 2**(j + 1), FS / 2**j)


def dwt_write(f, sl, counts, gain, baseline, out):
    for band, c in zip(DWT_KEEP, out):
        f["dwt"][band][sl] = c


paths["DWT"] = build_dataset("DWT", dwt_init, dwt_write, worker=dwt_worker)

  [DWT] batch 40/240    23.9 s


  [DWT] batch 80/240    46.5 s


  [DWT] batch 120/240    62.0 s


  [DWT] batch 160/240    79.5 s


  [DWT] batch 200/240    95.6 s


  [DWT] batch 240/240   109.1 s
[DWT] saved C:\Users\trist\Desktop\Research\Wrist-based EMG\Live\ProcessedDatasets\dwt_wrist\grabmyo_wrist_dwt.h5 (3.78 GB)


## 6. PSD dataset (FFT)

The PSD of the signal as released, in mV, is estimated with Welch's method over the whole 5 s trial: the trial is split into 1024-sample (0.5 s) Hann-windowed segments with 50 % overlap, each segment is FFT'd, and the periodograms are averaged. This gives 513 one-sided bins from 0 to 1024 Hz at 2 Hz resolution, the same settings as Section 4 of `grabmyo_sample_datapoint.ipynb`.

**Note:** Only the 246 bins from 10 to 500 Hz are stored due to band selection. This means the bins below 10 Hz (residual drift and motion, ~40 dB down after the authors' filter) and above 500 Hz (filter roll-off and noise) are not saved into the dataset.

| HDF5 key | Shape | dtype |
|---|---|---|
| `psd` | (15351, 12, 246) | float32, mV²/Hz |
| `freqs` | (246,) | float64, Hz (10, 12, …, 500) |

In [9]:
def psd_init(f):
    write_common_metadata(f, "GRABMyo wrist sEMG: Welch PSD of the signal as released")
    f.attrs.update(units="mV^2/Hz", method="scipy.signal.welch", window="hann",
                   nperseg=PSD_NPERSEG, noverlap=PSD_NOVERLAP, scaling="density", detrend="constant",
                   band_Hz=PSD_BAND, band_selection="only bins with 10 <= f <= 500 Hz stored (246 of 513)")
    f.create_dataset("freqs", data=PSD_FREQS)
    f.create_dataset("psd", (N_REC, N_CH, N_FREQ), "float32", chunks=(16, N_CH, N_FREQ))


def psd_write(f, sl, counts, gain, baseline, out):
    f["psd"][sl] = out[0]


paths["PSD"] = build_dataset("PSD", psd_init, psd_write, worker=psd_worker)

  [PSD] batch 40/240    26.5 s


  [PSD] batch 80/240    49.7 s


  [PSD] batch 120/240    71.2 s


  [PSD] batch 160/240    96.3 s


  [PSD] batch 200/240   118.4 s


  [PSD] batch 240/240   138.1 s
[PSD] saved C:\Users\trist\Desktop\Research\Wrist-based EMG\Live\ProcessedDatasets\psd_wrist\grabmyo_wrist_psd.h5 (0.18 GB)


## 7. Validation against the sample-data-point method

This cell recomputes a few trials (including the one in `grabmyo_sample_datapoint.ipynb`) directly, using `wfdb.rdrecord` followed by the same DWT and Welch PSD, and checks all three datasets:

- **Raw:** the stored counts convert back to exactly the `wfdb` signal in mV.
- **DWT:** the stored bands (A5, D5–D2) match `pywt.wavedec` within float32 precision, and rebuild the signal restricted to 0–512 Hz. It also reports how much of each trial's energy the stored bands keep.
- **PSD:** the stored 10–500 Hz bins match `scipy.signal.welch` within float32 precision, on the stored frequency grid.
- **Index:** all three files list the same trials in the same order.

In [10]:
check = index.index[(index.session == 1) & (index.participant == 1) & (index.gesture == 16) & (index.trial == 1)].tolist()
check += index.sample(4, random_state=1).index.tolist()

kept_energy = []
with h5py.File(paths["Raw"]) as fr, h5py.File(paths["DWT"]) as fd, h5py.File(paths["PSD"]) as fp:
    for col in ["session", "participant", "gesture", "trial", "record"]:
        assert (fr[col][:] == fd[col][:]).all() and (fr[col][:] == fp[col][:]).all(), col
    assert fd.attrs["window_samples"] == DWT_WINDOW
    dwt = fd["dwt"]
    assert set(dwt.keys()) == set(DWT_KEEP), sorted(dwt.keys())
    np.testing.assert_array_equal(fp["freqs"][:], PSD_FREQS)
    for i in check:
        x = wfdb.rdrecord(str(DATA_ROOT / index.path[i]), channel_names=WRIST).p_signal.T     # (12, N) mV
        raw_mv = (fr["emg"][i] - fr["baseline"][i][:, None]) / fr["gain"][i][:, None]
        np.testing.assert_allclose(raw_mv, x, rtol=0, atol=1e-12)

        coeffs = pywt.wavedec(x, WAVELET, mode=DWT_MODE, level=DWT_LEVEL, axis=-1)
        for band, c in zip(DWT_BANDS, coeffs):
            if band in DWT_KEEP:
                np.testing.assert_allclose(dwt[band][i], c, rtol=1e-6, atol=1e-6)           # float32 storage
        # Stored bands + zeros for D1 rebuild the signal restricted to 0-512 Hz
        x_rec = pywt.waverec([dwt[b][i] if b in DWT_KEEP else np.zeros_like(c) for b, c in zip(DWT_BANDS, coeffs)],
                             WAVELET, mode=DWT_MODE, axis=-1)
        x_ref = pywt.waverec([c if b in DWT_KEEP else np.zeros_like(c) for b, c in zip(DWT_BANDS, coeffs)],
                             WAVELET, mode=DWT_MODE, axis=-1)
        np.testing.assert_allclose(x_rec, x_ref, rtol=0, atol=1e-5)
        e = np.array([(c ** 2).sum() for c in coeffs])
        kept_energy.append(e[[DWT_BANDS.index(b) for b in DWT_KEEP]].sum() / e.sum())

        p = signal.welch(x, fs=FS, nperseg=PSD_NPERSEG, noverlap=PSD_NOVERLAP, window="hann", axis=-1)[1]
        np.testing.assert_allclose(fp["psd"][i], p[..., PSD_KEEP], rtol=1e-5, atol=1e-12)

kept_energy = 100 * np.array(kept_energy)
print(f"All {len(check)} checked trials match the reference in all three datasets:")
print("  Raw: exact;  DWT (A5, D5-D2): within float32 precision, rebuilds the 0-512 Hz signal;  "
      "PSD (10-500 Hz bins): within float32 precision.")
print("  The trial index is identical across the three files.")
print(f"  Energy kept by the stored DWT bands: median {np.median(kept_energy):.1f} % "
      f"(min {kept_energy.min():.1f} %, max {kept_energy.max():.1f} %)")

All 5 checked trials match the reference in all three datasets:
  Raw: exact;  DWT (A5, D5-D2): within float32 precision, rebuilds the 0-512 Hz signal;  PSD (10-500 Hz bins): within float32 precision.
  The trial index is identical across the three files.
  Energy kept by the stored DWT bands: median 98.4 % (min 97.5 %, max 99.5 %)
